<a id="Route-Generator"></a>
<h3 style="color:#1f77b4;">Route Generator</h3>
<h5 style="color:#1fb4a7;">Meter reading route builder using WA GNAF address data</h5>

<a id="Config"></a>
<h3 style="color:#1f77b4;">Config</h3>

In [31]:
import random
import pandas as pd


base_path = r"C:\Users\thoma\Documents\WA address GNAF"
output_path = r"C:\Users\thoma\Documents\balga_r_deadwalk.csv"


# (street, suburb, property_type)
streets_input = [
    ("STEDHAM", "BALGA", "R"),
    ("FITZROY", "BALGA", "R"),
    ("BREDE", "BALGA", "R"),
    ("ST KILDA", "BALGA", "R"),
    ("GRINSTEAD", "BALGA", "R"),
    ("MENTONE", "BALGA", "R"),

]


# # (street, suburb, property_type)
# streets_input = [
#     ("HILLIER", "HAMERSLEY", "R"),
#     ("DON", "HAMERSLEY", "R"),
#     ("FARMAN", "HAMERSLEY", "R"),
#     ("AVRO", "HAMERSLEY", "R"),
#     ("VICKERS", "HAMERSLEY", "R"),
# ]

seed = 42

<a id="Remote-Filter"></a>
<h3 style="color:#1f77b4;">Remote-Read Filter</h3>
<h5 style="color:#1fb4a7;">Simulate gaps in a meter reading walk</h5>

In [32]:
    # Replicate the pattern of remote reading impact on routes.
def filter_remote_pattern(group, seed_value):
    group = group.sort_values("number").copy()
    rng = random.Random(seed_value)
    keep = []
    state = "read"

    # Alternates between adding random batches of trues (1–5) 
    # and falses (1–8) until theres one value per row, then 
    # uses that list to keep or drop rows.
    # This simulates the clustered feel, where meter exchanges 
    # are done in batches. Electrician does 6 on a street.

    while len(keep) < len(group):
        if state == "read":
            run = rng.choice([1, 2, 3, 4, 5])
            keep.extend([True] * run)
            state = "skip"
        else:
            run = rng.choice([1, 2, 3, 4, 5, 6, 7, 8])
            keep.extend([False] * run)
            state = "read"

    return group[keep[:len(group)]]

<a id="Data-Acquisition"></a>
<h5 style="color:#1fb4a7;">Loading GNAF datasets</h5>

Converted to parquet seperatley for faster processing in python

In [33]:
address = pd.read_parquet(base_path + "\\WA_ADDRESS_DETAIL_psv.parquet")
geo = pd.read_parquet(base_path + "\\WA_ADDRESS_DEFAULT_GEOCODE_psv.parquet")
streets = pd.read_parquet(base_path + "\\WA_STREET_LOCALITY_psv.parquet")
locality = pd.read_parquet(base_path + "\\WA_LOCALITY_psv.parquet")

address.head()

,ADDRESS_DETAIL_PID,DATE_CREATED,DATE_LAST_MODIFIED,DATE_RETIRED,BUILDING_NAME,LOT_NUMBER_PREFIX,LOT_NUMBER,LOT_NUMBER_SUFFIX,FLAT_TYPE_CODE,FLAT_NUMBER_PREFIX,...,ALIAS_PRINCIPAL,POSTCODE,PRIVATE_STREET,LEGAL_PARCEL_ID,CONFIDENCE,ADDRESS_SITE_PID,LEVEL_GEOCODED_CODE,PROPERTY_PID,GNAF_PROPERTY_PID,PRIMARY_SECONDARY
0,GAWA_147148799,2004-04-29,2021-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,P,6167,NaN,P020818/145,2,147138600,7,NaN,129240.0,NaN
1,GAWA_147213598,2004-04-29,2021-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,P,6167,NaN,D097064/203,2,147201738,7,NaN,1229471.0,NaN
2,GAWA_147213595,2004-04-29,2021-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,P,6167,NaN,D097064/200,1,147201735,7,NaN,1229469.0,NaN
3,GAWA_147213596,2004-04-29,2021-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,P,6167,NaN,D097064/201,2,147201736,7,NaN,1229468.0,NaN
4,GAWA_147213597,2004-04-29,2021-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,P,6167,NaN,D097064/202,1,147201737,7,NaN,1229470.0,NaN


<a id="Trim-Columns"></a>
<h5 style="color:#1fb4a7;">Trim Columns</h5>
Trim to only needed columns

In [34]:
streets = streets[["STREET_LOCALITY_PID", "STREET_NAME", "STREET_TYPE_CODE", "LOCALITY_PID"]].rename(
    columns={"LOCALITY_PID": "STREET_LOCALITY_LOCALITY_PID"}
)

locality = locality[["LOCALITY_PID", "LOCALITY_NAME", "PRIMARY_POSTCODE"]]

<a id="Merge"></a>
<h5 style="color:#1fb4a7;">Merge</h5>

Bringing the required fields from each .psv into one dataframe

In [35]:
df = (
    address
    .merge(geo, on="ADDRESS_DETAIL_PID", how="inner")
    .merge(streets, on="STREET_LOCALITY_PID", how="left")
    .merge(locality, left_on="STREET_LOCALITY_LOCALITY_PID", right_on="LOCALITY_PID", how="left")
)

print(f"{len(df):,} rows after merge")

df.columns

1,666,702 rows after merge


Index(['ADDRESS_DETAIL_PID', 'DATE_CREATED_x', 'DATE_LAST_MODIFIED',
       'DATE_RETIRED_x', 'BUILDING_NAME', 'LOT_NUMBER_PREFIX', 'LOT_NUMBER',
       'LOT_NUMBER_SUFFIX', 'FLAT_TYPE_CODE', 'FLAT_NUMBER_PREFIX',
       'FLAT_NUMBER', 'FLAT_NUMBER_SUFFIX', 'LEVEL_TYPE_CODE',
       'LEVEL_NUMBER_PREFIX', 'LEVEL_NUMBER', 'LEVEL_NUMBER_SUFFIX',
       'NUMBER_FIRST_PREFIX', 'NUMBER_FIRST', 'NUMBER_FIRST_SUFFIX',
       'NUMBER_LAST_PREFIX', 'NUMBER_LAST', 'NUMBER_LAST_SUFFIX',
       'STREET_LOCALITY_PID', 'LOCATION_DESCRIPTION', 'LOCALITY_PID_x',
       'ALIAS_PRINCIPAL', 'POSTCODE', 'PRIVATE_STREET', 'LEGAL_PARCEL_ID',
       'CONFIDENCE', 'ADDRESS_SITE_PID', 'LEVEL_GEOCODED_CODE', 'PROPERTY_PID',
       'GNAF_PROPERTY_PID', 'PRIMARY_SECONDARY', 'ADDRESS_DEFAULT_GEOCODE_PID',
       'DATE_CREATED_y', 'DATE_RETIRED_y', 'GEOCODE_TYPE_CODE', 'LONGITUDE',
       'LATITUDE', 'STREET_NAME', 'STREET_TYPE_CODE',
       'STREET_LOCALITY_LOCALITY_PID', 'LOCALITY_PID_y', 'LOCALITY_NAME',
       

<a id="Clean"></a>
<h5 style="color:#1fb4a7;">Clean</h5>

Renaming columns to keep it clean, making more readable.

In [36]:
df = df.dropna(subset=["NUMBER_FIRST", "STREET_NAME", "LOCALITY_NAME", "POSTCODE", "LATITUDE", "LONGITUDE"]).copy()

df = df.rename(columns={
    "FLAT_NUMBER": "unit",
    "NUMBER_FIRST": "number",
    "STREET_NAME": "street",
    "STREET_TYPE_CODE": "type",
    "LOCALITY_NAME": "suburb",
    "POSTCODE": "postcode",
    "LATITUDE": "lat",
    "LONGITUDE": "lon"
})

df["number"] = df["number"].astype(int)
df["unit"] = df["unit"].astype("Int64")

df = df[["unit", "number", "street", "type", "suburb", "postcode", "lat", "lon"]].copy()

for col in ["street", "type", "suburb"]:
    df[col] = df[col].astype(str).str.upper().str.strip()

print(f"{len(df):,} rows after cleaning")

df

1,623,419 rows after cleaning


,unit,number,street,type,suburb,postcode,lat,lon
0,<NA>,916,ANKETELL,ROAD,ANKETELL,6167,-32.210050,115.879710
1,<NA>,107,BATTERSBY,ROAD,ANKETELL,6167,-32.218600,115.872010
2,<NA>,110,BATTERSBY,ROAD,ANKETELL,6167,-32.218620,115.867380
3,<NA>,96,BATTERSBY,ROAD,ANKETELL,6167,-32.217780,115.867380
4,<NA>,97,BATTERSBY,ROAD,ANKETELL,6167,-32.217760,115.872010
...,...,...,...,...,...,...,...,...
1666697,<NA>,16,BRAND,STREET,COOROW,6515,-29.882201,116.018572
1666698,<NA>,7,TAROT,GROVE,BALDIVIS,6171,-32.333283,115.794139
1666699,7,198,KENT,STREET,ROCKINGHAM,6168,-32.263820,115.746745
1666700,421,1,AIRLIE,STREET,CLAREMONT,6010,-31.992993,115.767525


<a id="Filter"></a>
<h5 style="color:#1fb4a7;">Filter to Suburb + Streets</h5>

Filter to just suburb and street inputs

In [37]:
# Build a lookup from streets_input for property_type
property_lookup = {}
for street_name, suburb_name, prop_type in streets_input:
    property_lookup[(street_name.upper(), suburb_name.upper())] = prop_type

# Filter to matching street+suburb pairs
mask = pd.Series(False, index=df.index)
for street_name, suburb_name, prop_type in streets_input:
    mask |= (df["street"] == street_name.upper()) & (df["suburb"] == suburb_name.upper())

matches = df[mask].copy()

# Assign property_type from lookup
property_types = []
for index, row in matches.iterrows():
    key = (row["street"], row["suburb"])
    property_types.append(property_lookup.get(key, ""))

matches["property_type"] = property_types

print(f"{len(matches)} addresses matched before remote filter")

352 addresses matched before remote filter


<a id="Apply-Remote-Filter"></a>
<h5 style="color:#1fb4a7;">Apply Remote-Read Filter</h5>

In [38]:
filtered = []
for (street_name, suburb_name), group in matches.groupby(["street", "suburb"]):
    street_seed = seed + sum(ord(c) for c in street_name + suburb_name)
    filtered.append(filter_remote_pattern(group, street_seed))

matches = pd.concat(filtered, ignore_index=True)
matches = matches.sort_values(["suburb", "street", "number", "unit"]).reset_index(drop=True)

print(f"{len(matches)} addresses after remote filter")
matches.head(10)

151 addresses after remote filter


,unit,number,street,type,suburb,postcode,lat,lon,property_type
0,<NA>,1,BREDE,PLACE,BALGA,6061,-31.856888,115.827311,R
1,<NA>,4,BREDE,PLACE,BALGA,6061,-31.856650,115.827861,R
2,<NA>,1,FITZROY,PLACE,BALGA,6061,-31.857303,115.827093,R
3,<NA>,1,FITZROY,PLACE,BALGA,6061,-31.857303,115.827093,R
4,<NA>,2,FITZROY,PLACE,BALGA,6061,-31.857354,115.826648,R
5,<NA>,3,FITZROY,PLACE,BALGA,6061,-31.857521,115.827091,R
6,<NA>,4,FITZROY,PLACE,BALGA,6061,-31.857600,115.826568,R
7,<NA>,7,FITZROY,PLACE,BALGA,6061,-31.858181,115.827048,R
8,<NA>,9,FITZROY,PLACE,BALGA,6061,-31.858031,115.826863,R
9,<NA>,1,GRINSTEAD,WAY,BALGA,6061,-31.857320,115.828430,R


<a id="Export"></a>
<h3 style="color:#1f77b4;">Export</h3>

In [39]:
matches = matches[["unit", "number", "street", "type", "suburb", "postcode", "lat", "lon", "property_type"]]
matches.to_csv(output_path, index=False)
print(f"{len(matches)} addresses exported to {output_path}")

151 addresses exported to C:\Users\thoma\Documents\balga_r_deadwalk.csv
